# Pipeline End-to-End: Prédiction de Consommation Électrique en France

## Nettoyage et Fusion Données Météo + RTE

Ce notebook démontre l'utilisation du pipeline complet pour :
1. Charger et nettoyer les données météorologiques (CSV)
2. Charger et nettoyer les données RTE (Excel)
3. Fusionner les datasets sur une clé temporelle
4. Gérer les valeurs manquantes et qualité des données

In [ ]:
import sys
from pathlib import Path
import logging

# Ajouter le répertoire scripts au path
sys.path.insert(0, str(Path('.').resolve()))

from scripts.data_pipeline import (
    EnergyDataPipeline,
    MeteoDataPipeline,
    RTEDataPipeline,
    DataFusionPipeline,
    DataQualityPipeline
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports réussis!")

## Configuration des chemins

In [ ]:
# Chemins de données
meteo_path = Path("./data/Data_Climat")
rte_path = Path("./data/Data_eCO2")
output_path = Path("./output")

# Créer les répertoires s'ils n'existent pas
output_path.mkdir(parents=True, exist_ok=True)

print(f"Chemin météo: {meteo_path}")
print(f"Chemin RTE: {rte_path}")
print(f"Chemin output: {output_path}")

## Exécution du Pipeline Complet

In [ ]:
# Créer et exécuter le pipeline
pipeline = EnergyDataPipeline(
    meteo_path=meteo_path,
    rte_path=rte_path,
    output_path=output_path
)

# Exécuter avec sauvegarde des fichiers intermédiaires
df_final = pipeline.run(
    meteo_date_col='date',  # À adapter selon votre nom de colonne
    rte_date_col='date',    # À adapter selon votre nom de colonne
    save_intermediate=True
)

## Exploration du Dataset Final

In [ ]:
print("\n" + "="*80)
print("STATISTIQUES DESCRIPTIVES")
print("="*80)
print(f"\nShape: {df_final.shape}")
print(f"\nInfo:")
df_final.info()
print(f"\nPremières lignes:")
print(df_final.head())

In [ ]:
print("\nStatistiques numériques:")
print(df_final.describe())

## Analyse de la Couverture Temporelle

In [ ]:
# Supposer que 'date' est la colonne temporelle
# À adapter selon votre structure réelle
if 'date' in df_final.columns:
    df_final['date'] = pd.to_datetime(df_final['date'])
    
    print("\nCouverture temporelle:")
    print(f"  Date min: {df_final['date'].min()}")
    print(f"  Date max: {df_final['date'].max()}")
    print(f"  Durée: {df_final['date'].max() - df_final['date'].min()}")
    
    # Vérifier les gaps temporels
    df_sorted = df_final.sort_values('date')
    time_diff = df_sorted['date'].diff()
    print(f"\nGaps temporels (min):")
    print(f"  Mode: {time_diff.mode()[0]}")
    print(f"  Min: {time_diff.min()}")
    print(f"  Max: {time_diff.max()}")

## Visualisation de la Qualité des Données

In [ ]:
# Heatmap des valeurs manquantes
fig, ax = plt.subplots(figsize=(14, 6))

# Afficher les colonnes avec des valeurs manquantes
missing_data = df_final.isnull().sum()
missing_percent = (missing_data / len(df_final)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing_data[missing_data > 0].index,
    'Valeurs manquantes': missing_data[missing_data > 0].values,
    'Pourcentage': missing_percent[missing_data > 0].values
}).sort_values('Pourcentage', ascending=False)

if len(missing_df) > 0:
    ax.barh(missing_df['Colonne'], missing_df['Pourcentage'], color='coral')
    ax.set_xlabel('Pourcentage de valeurs manquantes (%)')
    ax.set_title('Analyse des Valeurs Manquantes par Colonne')
    plt.tight_layout()
    plt.show()
    print(missing_df.to_string(index=False))
else:
    print("✓ Aucune valeur manquante détectée!")

## Analyse des Codes Départements

In [ ]:
# Analyser les codes départements
if 'code_dept' in df_final.columns:
    print("\nCodes Départements présents:")
    dept_counts = df_final['code_dept'].value_counts().sort_index()
    print(dept_counts)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    dept_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Nombre de Lignes par Code Département')
    ax.set_xlabel('Code Département')
    ax.set_ylabel('Nombre de Lignes')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Export du Dataset Final

In [ ]:
# Sauvegarder le dataset final
output_file = output_path / "dataset_final_clean.csv"
df_final.to_csv(output_file, index=False)
print(f"✓ Dataset final sauvegardé: {output_file}")
print(f"  Taille: {output_file.stat().st_size / (1024*1024):.2f} MB")

# Sauvegarde en Parquet (plus compressé et optimisé pour Pandas)
output_file_parquet = output_path / "dataset_final_clean.parquet"
df_final.to_parquet(output_file_parquet, compression='snappy', index=False)
print(f"✓ Dataset final sauvegardé (Parquet): {output_file_parquet}")
print(f"  Taille: {output_file_parquet.stat().st_size / (1024*1024):.2f} MB")

## Rapport de Synthèse

In [ ]:
print("\n" + "="*80)
print("RAPPORT DE SYNTHÈSE DU PIPELINE")
print("="*80)

summary = f"""
📊 RÉSULTATS FINAUX

Dimensions:
  • Nombre de lignes: {df_final.shape[0]:,}
  • Nombre de colonnes: {df_final.shape[1]}

Colonnes:
  • {', '.join(df_final.columns.tolist())}

Qualité des Données:
  • Valeurs manquantes totales: {df_final.isnull().sum().sum()}
  • Pourcentage de complétude: {(1 - df_final.isnull().sum().sum() / (df_final.shape[0] * df_final.shape[1])) * 100:.2f}%

Types de données:
  • Colonnes numériques: {len(df_final.select_dtypes(include=[np.number]).columns)}
  • Colonnes texte/catégorie: {len(df_final.select_dtypes(include=['object', 'category']).columns)}
  • Colonnes datetime: {len(df_final.select_dtypes(include=['datetime64']).columns)}

✓ Pipeline exécuté avec succès!
"""

print(summary)